In [1658]:
from discovery_child_development import PROJECT_DIR
import pandas as pd
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'
PATH_TO_DATASET = ENRICHED_DATA_DIR / 'openalex_patents_relevance_labels_only_relevant.csv'

# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

38


In [1659]:
# Load the data with texts
text_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
)
len(text_df)

51234

In [1669]:
dfs = []
for topic in topics:
    keywords = topics_dict[topic]["filtering_keywords"]
    df = (
        pd.read_csv(ENRICHED_DATA_DIR / f'taxonomy_cat/taxonomy_cat_predictions_{topic}.csv')
        .merge(text_df[['id', 'text']], on='id', how='left')
    )
    keyword_hits = (
        df.text
        .str.lower()
        .str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)
        .str.contains("|".join(keywords))
    )
    df = df[keyword_hits]
    dfs.append(df)

labelled_df = (
    pd.concat(dfs, ignore_index=True)
    .query("prediction==1.0")
    .groupby("id")
    .agg(topics = ("topic", list))
    .reset_index()
    .astype({"topics": str})
)

text_labelled_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
    .merge(labelled_df, on="id", how="left")
)

In [1653]:
# Papers with no labels
text_labelled_df.topics.isnull().sum()

1587

In [1638]:
topic = "ai2"
text_labelled_df.topics.str.contains(topic).count()

# keywords = ["prenatal", "newborn", "birth", "maternal", "neonate"]
keywords = [" ai ", "artificial intelligence", "data", "science", "machine learning", "deep learning", "neural network", "algorithm", "model", "predict", "smart", "intelligent", "monitor", "sensor", "information", "convolut", "graph", "robot", "chatbot", "natural language processing", " nlp ", "automat"]
keywords = [word.lower() for word in keywords]

df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    & (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == False)
]

print(len(df))

/var/folders/zq/6141y2zd39x441hglsbcvwxm0000gp/T/ipykernel_29172/2894638532.py:10: FutureWarning: The default value of regex will change from True to False in a future version.
  & (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == False)


37


In [1643]:

df.sample().iloc[0].text

'User control system and method for infant care apparatus. An infant care system comprises a platform configured to support an infant and to receive an applied force on the platform. The infant care system also comprises at least two load cells connected to the platform, each of the two load cells configured to receive at least a portion of the applied force on the platform and generate a signal indicative of the portion of applied force received. The infant care system also comprises a processor configured to receive and analyze the signals from the load cells and perform a function based on the analyzed signals.'

In [988]:
# keywords = keywords + ["setting", "centre", "center"]
# df2 = text_labelled_df[
#     (text_labelled_df.topics.str.contains(topic) == True)
#     & (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)
# ]
# print(len(df2))

/var/folders/zq/6141y2zd39x441hglsbcvwxm0000gp/T/ipykernel_29172/1831655078.py:4: FutureWarning: The default value of regex will change from True to False in a future version.
  & (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)


9900


In [989]:
# # get the difference between df2 and df
# df = df2[~df2.id.isin(df.id)]
# len(df)

1380

In [343]:
topic = "rct"
text_labelled_df.topics.str.contains(topic).count()

keywords = [" rct ", "random", "random"]
keywords = [word.lower() for word in keywords]


df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)
]
print(len(df))
df.sample(1).iloc[0].text

/var/folders/zq/6141y2zd39x441hglsbcvwxm0000gp/T/ipykernel_29172/1111074148.py:11: FutureWarning: The default value of regex will change from True to False in a future version.
  (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)


2421


'Adult Cardiovascular Health Risk and Cardiovascular Phenotypes of Prematurity. Survival of extremely low birth weight (ELBW) infants has improved dramatically over the past 20 years. 1 Watkins P.L. Dagle J.M. Bell E.F. Colaizy T.T. Outcomes at 18 to 22 months of corrected age for infants born at 22 to 25 weeks of gestation in a center practicing active management. J Pediatr. 2020; 217: 52-58.e1 Abstract Full Text Full Text PDF PubMed Google Scholar The nature of these advances is multifactorial and relates to increased appreciation of organ vulnerability, improved understanding of disease mechanisms, enhanced diagnostic precision, and improved therapeutic options. Unfortunately, the advances that underlie enhanced survival do not guarantee avoidance of neonatal morbidity or adverse long-term outcomes. In addition, life-saving treatments may have unintended negative effects on organ development and performance. Recent evidence highlights the relationship between prematurity and increas

In [309]:
topic = "emotional"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["social", "emotional", "awareness", "relationship", "relate"]
keywords = [word.lower() for word in keywords]


df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)
]
print(len(df))
df.sample(1).iloc[0].text

/var/folders/zq/6141y2zd39x441hglsbcvwxm0000gp/T/ipykernel_29172/1805871294.py:11: FutureWarning: The default value of regex will change from True to False in a future version.
  (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)


6918


'Professionalisation of childcare assistants in early childhood education and care (ECEC): pathways towards qualification. There is growing evidence among researchers and international organisations that the quality of early childhood education and care (ECEC), and ultimately the outcomes for children and families - especially disadvantaged ones - is dependent on well-educated and competent staff, and that a lack of higher pre-service training can be partly compensated by in-service training of a sufficient intensity and length. This report comes from the Network of Experts on Social Aspects of Education and Training (NESET) II, set up by the European Commission to work on the social dimension of education and training.'

In [297]:
topic = "communication"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["language", "communicat", "speech", "ling"]
keywords = [word.lower() for word in keywords]


df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == False)
]
print(len(df))
df.sample(1).iloc[0].text

/var/folders/zq/6141y2zd39x441hglsbcvwxm0000gp/T/ipykernel_29172/4050333609.py:11: FutureWarning: The default value of regex will change from True to False in a future version.
  (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == False)


3452


"The Chicago School Readiness Project: Examining the long-term impacts of an early childhood intervention. The current paper reports long-term treatment impact estimates for a randomized evaluation of an early childhood intervention designed to promote children's developmental outcomes and improve the quality of Head Start centers serving high-violence and high-crime areas in inner-city Chicago. Initial evaluations of end-of-preschool data reported that the program led to reductions in child behavioral problems and gains in measures of executive function and academic achievement. For this report, we analyzed adolescent follow-up data taken 10 to 11 years after program completion. We found evidence that the program had positive long-term effects on students' executive function and grades, though effects were somewhat imprecise and dependent on the inclusion of baseline covariates. Results also indicated that treated children had heightened sensitivity to emotional stimuli, and we found 

In [288]:
topic = "physical"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["motor", "physical", "spatial", "walk", "move"]
keywords = [word.lower() for word in keywords]


df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)
]
print(len(df))
df.sample(1).iloc[0].text

/var/folders/zq/6141y2zd39x441hglsbcvwxm0000gp/T/ipykernel_29172/2611444374.py:11: FutureWarning: The default value of regex will change from True to False in a future version.
  (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)


3460


'Objective Measurement of Fusional Vergence Ranges and Heterophoria in Infants and Preschool Children. Purpose: Binocular alignment typically includes motor fusion compensating for heterophoria. This study evaluated heterophoria and then accommodation and vergence responses during measurement of fusional ranges in infants and preschoolers. Methods: Purkinje image eye tracking and eccentric photorefraction (MCS PowerRefractor) were used to record the eye alignment and accommodation of uncorrected infants (n = 17; 3–5 months old), preschoolers (n = 19; 2.5–5 years), and naïve functionally emmetropic adults (n = 14; 20–32 years; spherical equivalent [SE], +1 to −1 diopters [D]). Heterophoria was derived from the difference between monocular and binocular alignments while participants viewed naturalistic images at 80 cm. The presence or absence of fusion was then assessed after base-in (BI) and base-out (BO) prisms (2–40 prism diopters [pd]) were introduced. Results: Mean (±SD) SE refracti

In [251]:
topic = "arts"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["creativ", " art ", " arts ", "imagin", " sing", "music", "drama"]
keywords = [word.lower() for word in keywords]


df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    # replace string non alphanumeric characters with space
    (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)
]
print(len(df))
df.sample(1).iloc[0].text

/var/folders/zq/6141y2zd39x441hglsbcvwxm0000gp/T/ipykernel_29172/576649565.py:12: FutureWarning: The default value of regex will change from True to False in a future version.
  (text_labelled_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ').str.contains("|".join(keywords)) == True)


2129


'A programming device and method for controlling based on resistance. The invention discloses a programming device and method for control based on resistance, which overcomes the problem of complicated and inconvenient use of the upper computer in the prior art. , There are a number of building block slots on the control panel of the shell, a reading circuit module is placed under the building block slot, and a resistance circuit is arranged inside the instruction building block. value, each reading circuit module and the corresponding resistance circuit form a voltage dividing circuit module, and the instruction building block is detachably installed in the building block slot. The invention uses several resistance circuits to generate different currents, and by presetting the reading resistance module, the I/O signal is converted into a control logic command and sent to the actuator for execution through Bluetooth communication. Young children get rid of the tediousness of using uppe

In [199]:
topic = "literacy"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["read", "writ", "letter", "handwriting", "liter", "phonic", "language", "ling"]
keywords = [word.lower() for word in keywords]

df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.contains("|".join(keywords)) == True)
]
print(len(df))
df.sample(1).iloc[0].text

2120


'Predictors of kindergarten‐reading performance for children with special needs: Do intervention intensity and service provider matter?. Abstract Over 1 million children under age 5 years who have special needs are served by early intervention and early childhood special education (EI/ECSE) intended to promote academic and non‐cognitive school readiness. Past research suggested these services may have null or negative effects on kindergarten‐reading skills, but these studies did not account for features of the services received by participants. Using data from the Early Childhood Longitudinal Study–Birth Cohort, this study explored the relations between kindergarten‐reading performance and EI/ECSE intensity and service provider. Analyses of a nationally representative sample of 550 participants (67% male, 63% White) indicated approximately 30% of the variance in kindergarten‐reading is explained by a combination of child characteristics, hours of service and type of providers. Results 

In [178]:
topic = "mathematics"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["math", "numer", "number", " stem ", "science", "scientific"]
# keywords = [word.lower() for word in keywords]

df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.contains("|".join(keywords)) == True)
]
print(len(df))
df.sample(1).iloc[0].text

1852


'Impact of early childcare on immigrant children’s educational performance. This paper investigates the impact of attending early childcare on immigrant children’s cognitive outcomes. Our analysis makes use of administrative data on the entire population of students in the fifth grade collected by the Italian Institute for the Evaluation of the Educational System (INVALSI) for school years 2014/2015 to 2016/2017, matched to unique administrative records on early childcare availability at the municipal level. Our identification strategy exploits cross-sectional and time series variations in the provision of early childcare services across Italian municipalities as an instrument for individual attendance. Our results indicate that the effect of early childcare attendance differs between native and immigrant children. Estimates show a positive and significant effect on the language test scores of immigrant children, with the effect being mostly driven by children with low-educated mothers

In [95]:
topic = "mobile"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["mobile", "phone", "smartphone", "smart phone", " app ", " apps ", "tablet", "iphone", " ipad ", "android", "screen time", "screentime", "touchscreen", "touch screen"]


df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.contains("|".join(keywords)) == True)
]

df.sample(1).iloc[0].text

'A multimedia interactive system and method based on early childhood education. The invention discloses a multimedia interactive system and method based on early childhood education. Including: picture content recognition, the picture comes from a touchable handheld device or a touchable desktop device, a static picture drawn by Tuya software; according to the recognized content, make part materials; according to the recognized content, from the animation library Extract the prefabricated animation; assign the produced material to the corresponding part of the prefabricated animation to ensure that the result is what you draw; display the final intelligent animation on the screen; connect the Kinect computer external device, and perform voice interaction with the intelligent animation through Kinect; Kinect performs gesture interaction with smart animation; connects to the touch screen, and can interact with smart animation through the touch screen. The invention allows children to giv

In [129]:
topic = "ar_vr"
text_labelled_df.topics.str.contains(topic).count()

keywords = ["augmented reality", "virtual reality", "mixed reality", "immersive", " AR ", " VR ", "virtual", "AR/VR"]
keywords = [word.lower() for word in keywords]

df = text_labelled_df[
    (text_labelled_df.topics.str.contains(topic) == True)
    &
    (text_labelled_df.text.str.lower().str.contains("|".join(keywords)) == False)
]
print(len(df))
df.sample(1).iloc[0].text

82


'An intelligent terminal robot with multimedia interaction for early childhood education and learning. The utility model discloses a multimedia interactive intelligent terminal robot for children&#39;s early education and learning, comprising a central processing unit, an input end of the central processing unit is electrically connected with an information analysis module, and an input end of the information analysis module is electrically connected with a voice The processor, the input end of the speech processor is electrically connected with the speech receiving module, the other input end of the central processor is electrically connected with the mobile phone intelligent terminal, and the input end of the speech processor is electrically connected with the wireless terminal. By adding a voice processing module inside the robot, the utility model can transmit the voice signal to the multimedia terminal for screen projection display, and when the robot fails, it can be reminded by 

In [ ]:
# labelled_df.to_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_all.csv', index=False)